# 02_06 Profiling and Enrichment `passengers_sncb_2024` Dataset

In [1]:
from pathlib import Path

import pandas as pd

import infrabel_punctuality as ip

In [2]:
PATH_INTERMEDIATE = Path("../data/intermediate/")

# Table of Contents

- [6. 6. AVERAGE DAILY PASSENGER COUNT - SNCB 2024](#6-average-daily-passenger-count---sncb-2024)
- [6.1. Data Profiling](#61-data-profiling)
    - [6.1.1. Overview](#611-overview)
    - [6.1.2. Looking for Missing Values](#612-looking-for-missing-values)

- [6.2. Data Quality and Cleaning](#62-data-quality-and-cleaning)
    - [6.2.1. Renaming Columns](#621-renaming-columns)
    - [6.2.2. Profiling Missing Values](#622-profiling-missing-values)
    - [6.2.3. Handling Missing Values](#623-handling-missing-values)
    - [6.2.4. Converting Data Types](#624-converting-data-types)
    - [6.2.5. Harmonising Station Names](#625-harmonising-station-names)
        - [6.2.5.1. Profiling Station Names & Finding Orphans](#6251-profiling-station-names--finding-orphans)
        - [6.2.5.2. Separating Station DataFrames with and without IDs](#6252-separating-station-dataframes-with-and-without-ids)
        - [6.5.2.3. Resolving Orphan Station Names](#6523-resolving-orphan-station-names)
        - [6.5.2.4. Concatenating Passenger Count Average Datasets with IDs](#6524-concatenating-passenger-count-average-datasets-with-ids)
    
- [6.3. Export to Silver Layer](#63-export-to-silver-layer)

## 6. AVERAGE DAILY PASSENGER COUNT - SNCB 2024

This dataset is extracted from a SNCB PDF. It contains the **average number of boarding passengers per station**, segmented by **weekdays**, **Saturdays**, and **Sundays**.

It is based on the **SNCB annual one-week passenger count survey**, conducted in **October 2024**.  

It will be used to enrich the future `fact_punctuality` table and calculate the **new delay metric weighted by the average number of passengers per station** in the *03_03_building_fact_punctuality* notebook. It will not be permanently merged with `stations_cleaned` to enrich the `stations_dimension`, even though this might seem the most natural approach, in order to avoid costly calculations involving joins between the `stations` dimension and the fact table `punctuality` every time the new metric is used in Power BI.  

However, a `ptcarid` column containing the station IDs will be added to this dataset through a temporary merging with `stations_cleaned`, from which this column will be extracted, to facilitate the final merge with `punctuality_cleaned` and the `fact_punctuality` enrichment.

**Note**: This PDF dataset was the most recent SNCB passenger count available at the beginning of this project.  

## 6.1. Data Profiling

### 6.1.1. Overview

* This dataset does not have an ID column. To merge it with `punctuality_cleaned`, the only column possible is the **station names column**.

* The station names column of `passengers_avg_sncb_2024` appears to mix French and Dutch station names, depending on where the station is located. 

* The dataset has **554 unique station names**.

* It has no duplicated rows and, at first glance, no missing values. However, it appears that missing values in `passengers_avg_sncb_2024` are not `NaN` or `NA` values. They are replaced by the **`"-"` string character**.  

* The column headers contain problematic characters such as `"\n"`. 

**DECISION**

* Cleaning and renaming the column headers.

* Replacing the `"-"` by `<NA>` to handle the missing values.

* Converting the average count columns to `Float64`.

* Cleaning and standardizing the station names to prepare the merge with `stations_cleaned`.

* Once these steps are completed, the merge with `punctuality_cleaned` will require two additional steps:  

    * Since the station names in `passengers_avg_sncb_2024` are provided in French or in Dutch, and given that the `PTCAR_LG_NM_NL` column in `punctuality_cleaned` only contains the Dutch station names, an intermediate merge with `stations_cleaned` will be required to enrich `passengers_avg_sncb_2024` with the `ptcarid` column from `stations_cleaned`.  

    * As `passengers_avg_sncb_2024` provides three distinct passenger count averages - one for weekdays, one for Saturdays, and one for Sundays -, the enrichment of `punctuality_cleaned` will require assigning each `DATDEP` value to **the correct day category** in order to match each entry with the appropriate passenger count average. This final step will be performed in the *03_03_building_fact_punctuality* notebook.

In [3]:
passengers_avg_sncb_2024 = pd.read_parquet(PATH_INTERMEDIATE / "passengers_sncb_2024.parquet")
passengers_avg_sncb_2024.head()

,Station \nGare,Gem. instappende tijdens week \nMoy. montés en semaine,Gem. instappende op zaterdag\nMoy. montés le samedi,Gem. instappende op zondag\nMoy. montés le dimanche
0,AALST,5.883,2.220,1.872
1,AALST-KERREBROEK,15,-,-
2,AALTER,2.348,622,620
3,AARSCHOT,5.804,2.243,1.758
4,AARSELE,26,-,-


In [4]:
passengers_avg_sncb_2024.info()

<class 'pandas.DataFrame'>
RangeIndex: 554 entries, 0 to 553
Data columns (total 4 columns):
 #   Column                                                 Non-Null Count  Dtype
---  ------                                                 --------------  -----
 0   Station 
Gare                                          554 non-null    str  
 1   Gem. instappende tijdens week 
Moy. montés en semaine  554 non-null    str  
 2   Gem. instappende op zaterdag
Moy. montés le samedi     554 non-null    str  
 3   Gem. instappende op zondag
Moy. montés le dimanche     554 non-null    str  
dtypes: str(4)
memory usage: 27.2 KB


In [5]:
passengers_avg_sncb_2024.nunique()

0
Station \nGare                                            554
Gem. instappende tijdens week \nMoy. montés en semaine    452
Gem. instappende op zaterdag\nMoy. montés le samedi       343
Gem. instappende op zondag\nMoy. montés le dimanche       316
dtype: int64

In [6]:
passengers_avg_sncb_2024.isnull().sum()

0
Station \nGare                                            0
Gem. instappende tijdens week \nMoy. montés en semaine    0
Gem. instappende op zaterdag\nMoy. montés le samedi       0
Gem. instappende op zondag\nMoy. montés le dimanche       0
dtype: int64

In [7]:
print(passengers_avg_sncb_2024["Station \nGare"].duplicated().sum())
passengers_avg_sncb_2024.duplicated().sum()

0


np.int64(0)

### 6.1.2. Looking for Missing Values

Missing values in this dataset are encoded as `"-"` rather than `NaN` or `<NA>`.

In [8]:
missing_values = ["-"]

columns_to_check = [
    "Gem. instappende tijdens week \nMoy. montés en semaine",
    "Gem. instappende op zaterdag\nMoy. montés le samedi",
    "Gem. instappende op zondag\nMoy. montés le dimanche"
                    ]

In [9]:
print(passengers_avg_sncb_2024[columns_to_check].isin(missing_values).sum())


0
Gem. instappende tijdens week \nMoy. montés en semaine     2
Gem. instappende op zaterdag\nMoy. montés le samedi       45
Gem. instappende op zondag\nMoy. montés le dimanche       44
dtype: int64


## 6.2. Data Quality and Cleaning

### 6.2.1. Renaming Columns

As the original column headers extracted from the PDF contain problematic characters such as `" \n"` and are unnecessarily long, they are renamed before profiling and handling its missing values:  

* `'Station \nGare'` -> `"stations_sncb"`
* `'Gem. instappende tijdens week \nMoy. montés en semaine'` -> `"passenger_avg_weekdays"`  
* `'Gem. instappende op zaterdag\nMoy. montés le samedi'` -> `"passenger_avg_saturday"`   
* `'Gem. instappende op zondag\nMoy. montés le dimanche'` -> `"passenger_avg_sunday"` 

In [10]:
passengers_avg_sncb_2024.columns

Index(['Station \nGare',
       'Gem. instappende tijdens week \nMoy. montés en semaine',
       'Gem. instappende op zaterdag\nMoy. montés le samedi',
       'Gem. instappende op zondag\nMoy. montés le dimanche'],
      dtype='str', name=0)

In [11]:
passengers_avg_sncb_2024 = passengers_avg_sncb_2024.rename(columns={
    "Station \nGare" : "stations_sncb",
    "Gem. instappende tijdens week \nMoy. montés en semaine" : "passenger_avg_weekdays",
    "Gem. instappende op zaterdag\nMoy. montés le samedi" : "passenger_avg_saturday",
    "Gem. instappende op zondag\nMoy. montés le dimanche" : "passenger_avg_sunday"
})

In [12]:
passengers_avg_sncb_2024.columns

Index(['stations_sncb', 'passenger_avg_weekdays', 'passenger_avg_saturday',
       'passenger_avg_sunday'],
      dtype='str', name=0)

### 6.2.2. Profiling Missing Values

As shown in the section below:  

* **508** stations in the `passengers_avg_sncb_2024` dataset have no missing values.  

* **44** stations have missing values for weekend passenger count averages, but have passenger count averages available for weekdays.

* Only two stations - `MORTSEL-DEURNESTEENWEG` and `ZEEBRUGGE-STRAND` - do not have weekday passenger count average available:  
    * `MORTSEL-DEURNESTEENWEG` appears to be open to passengers only on Sundays.  
    * `ZEEBRUGGE-STRAND` appears to be open to passengers only during the weekends, both on Saturdays and Sundays.

In [13]:
columns_with_nan = ["passenger_avg_weekdays", "passenger_avg_saturday", "passenger_avg_sunday"]

nan_row_profile = pd.concat(
    [passengers_avg_sncb_2024[col].str.contains("-") for col in columns_with_nan], axis=1
)

nan_row_profile.value_counts()

passenger_avg_weekdays  passenger_avg_saturday  passenger_avg_sunday
False                   False                   False                   508
                        True                    True                     44
True                    True                    False                     1
                        False                   False                     1
Name: count, dtype: int64

In [14]:
open_every_day = nan_row_profile[~nan_row_profile.any(axis=1)].shape[0]
print(f"{open_every_day} stations open every day.")

open_weekdays = nan_row_profile[(~nan_row_profile["passenger_avg_weekdays"]) & 
             nan_row_profile["passenger_avg_saturday"] & 
             nan_row_profile["passenger_avg_sunday"]].shape[0]
print(f"{open_weekdays} stations open only on weekdays.")

open_sundays = nan_row_profile[(nan_row_profile["passenger_avg_weekdays"]) &
            nan_row_profile["passenger_avg_saturday"] &
            ~nan_row_profile["passenger_avg_sunday"]].shape[0]
print(f"{open_sundays} station{'s' if open_sundays != 1 else ''} open only on Sundays.")

open_saturdays = nan_row_profile[(nan_row_profile["passenger_avg_weekdays"]) &
            ~nan_row_profile["passenger_avg_saturday"] &
            nan_row_profile["passenger_avg_sunday"]].shape[0]
print(f"{open_saturdays} station{'s' if open_saturdays != 1 else ''} open only on Saturdays.")

open_weekends = nan_row_profile[(nan_row_profile["passenger_avg_weekdays"]) &
                ~nan_row_profile["passenger_avg_saturday"] &
                ~nan_row_profile["passenger_avg_sunday"]].shape[0]
print(f"{open_weekends} station{'s' if open_weekends != 1 else ''} open only on weekends.")

508 stations open every day.
44 stations open only on weekdays.
1 station open only on Sundays.
0 stations open only on Saturdays.
1 station open only on weekends.


In [15]:
nan_rows = (
    passengers_avg_sncb_2024.loc[
    (passengers_avg_sncb_2024["passenger_avg_weekdays"].str.contains("-")) | 
    (passengers_avg_sncb_2024["passenger_avg_saturday"].str.contains("-"))  | 
    (passengers_avg_sncb_2024["passenger_avg_sunday"].str.contains("-"))
    ].sort_values("passenger_avg_weekdays")
    )

nan_rows.sort_values("passenger_avg_saturday", ascending=False).head()

,stations_sncb,passenger_avg_weekdays,passenger_avg_saturday,passenger_avg_sunday
545,ZEEBRUGGE-STRAND,-,140,140
373,MORTSEL-DEURNESTEENWEG,-,-,105
347,MAUBRAY,10,-,-
203,GHLIN,102,-,-
252,HOFSTADE,104,-,-


### 6.2.3. Handling Missing Values

In the section below:

* The function `clean_df_string()` of the `infrabel_punctuality` package is used to clean the `str` columns containing `"-"`.

* Then, all the `"-"` `str` caracters are replaced by `pd.NA`. As there are no passenger counts for these day categories (weekdays, saturday or sunday), the missing values will be preserved as `NULL` in the SQL data warehouse.

In [16]:
columns_with_missing_values = [
    "passenger_avg_weekdays",
    "passenger_avg_saturday",
    "passenger_avg_sunday"
]

ip.clean_df_string(passengers_avg_sncb_2024, columns_with_missing_values)


,stations_sncb,passenger_avg_weekdays,passenger_avg_saturday,passenger_avg_sunday
0,AALST,5.883,2.220,1.872
1,AALST-KERREBROEK,15,-,-
2,AALTER,2.348,622,620
3,AARSCHOT,5.804,2.243,1.758
4,AARSELE,26,-,-
...,...,...,...,...
549,ZINGEM,402,135,103
550,ZOLDER,110,17,37
551,ZONHOVEN,60,14,37
552,ZOTTEGEM,4.360,1.186,999


In [17]:
passengers_avg_sncb_2024[columns_with_missing_values] = (
    passengers_avg_sncb_2024[columns_with_missing_values].replace("-", pd.NA)
)

In [18]:
passengers_avg_sncb_2024.isnull().sum()

0
stations_sncb              0
passenger_avg_weekdays     2
passenger_avg_saturday    45
passenger_avg_sunday      44
dtype: int64

### 6.2.4. Converting Data Types

* To allow conversion of `<NA>` values to `NULL` in the future **SQL Data Warehouse**, the `stations_sncb` column is converted to `string` data type.  

* To allow conversion of `<NA>` values to `NULL` in SQL, the `str` columns containing the **passenger count average** by weekdays (`passenger_avg_weekdays`), Saturdays (`passenger_avg_saturday`), and Sundays (`passenger_avg_sunday`) are converted to `Float64`.

* Then, these columns are checked to verify that they contain only integers without decimals and finally converted to `Int64`.

In [19]:
columns_to_convert = [
    "passenger_avg_weekdays",
    "passenger_avg_saturday",
    "passenger_avg_sunday"
]

passengers_avg_sncb_2024[columns_to_convert] = (
    passengers_avg_sncb_2024[columns_to_convert].replace({r"\." : ""}, regex=True)
    )

In [20]:
passengers_avg_sncb_2024.columns

Index(['stations_sncb', 'passenger_avg_weekdays', 'passenger_avg_saturday',
       'passenger_avg_sunday'],
      dtype='str', name=0)

In [21]:
passengers_avg_sncb_2024["stations_sncb"] = passengers_avg_sncb_2024["stations_sncb"].astype("string")
passengers_avg_sncb_2024[columns_to_convert] = passengers_avg_sncb_2024[columns_to_convert].astype("Float64")

In [22]:
passengers_avg_sncb_2024.info()

<class 'pandas.DataFrame'>
RangeIndex: 554 entries, 0 to 553
Data columns (total 4 columns):
 #   Column                  Non-Null Count  Dtype  
---  ------                  --------------  -----  
 0   stations_sncb           554 non-null    string 
 1   passenger_avg_weekdays  552 non-null    Float64
 2   passenger_avg_saturday  509 non-null    Float64
 3   passenger_avg_sunday    510 non-null    Float64
dtypes: Float64(3), string(1)
memory usage: 24.0 KB


In [23]:
col = passengers_avg_sncb_2024["passenger_avg_weekdays"]
(col % 1 == 0).all()

np.True_

In [24]:
col_1 = passengers_avg_sncb_2024["passenger_avg_saturday"]
(col_1 % 1 == 0).all()

np.True_

In [25]:
col_2 = passengers_avg_sncb_2024["passenger_avg_sunday"]
(col_2 % 1 == 0).all()

np.True_

In [26]:
passengers_avg_sncb_2024[columns_to_convert] = passengers_avg_sncb_2024[columns_to_convert].astype("Int64")

### 6.2.5. Harmonising Station Names

To enrich the future `punctuality` fact table, a `ptcarid` column must be added to `passengers_avg_sncb_2024` to enable the merge with `punctuality_cleaned`. This requires harmonising the stations names in `stations_sncb` with those in `stations_cleaned`, the **Infrabel reference dataset** for all station and passenger stop information, which contains the `ptcarid` column.  

The `stations_sncb` column presents **several inconsistencies**: non-standardised abbreviations, mixed French and Dutch names in the same column - sometimes within the same value (e.g. `"UKKEL/UCCLE CALEVOET"` instead of `"UCCLE-CALEVOET"` in French or `"UKKEL-KALEVOET"` in Dutch) - and other formatting irregularities. In contrast, `stations_cleaned` clearly separates French and Dutch station names into the `longfrenchname` and `longdutchname` columns, and contains no abbreviations.

### 6.2.5.1. Profiling Station Names & Finding Orphans

In the section below: 

* The `stations_for_validation` DataFrame is created from `stations_cleaned` by dropping irrelevant columns.  

* `passenges_avg_sncb_2024` and `stations_for_validation` are **merged** to assign the corresponding `ptcarid` values to each `passengers_avg_sncb_2024` stations.

* After the merge: 
    * **508** out of 554 stations are directly matched with their correct station ID.
    * **46** out of 554 stations are orphans with a missing station ID.  

* The 46 orphan station names are displayed at the end of this section to visually profile the different inconsistencies that will be resolved in the next sections.

**NOTE**: The merge is performed on `stations_sncb` and `longnamefrench` columns. `longnamefrench` was selected over `longnamedutch` after testing both: merging on `longnamedutch` generates 47 orphan stations - one more than `longnamefrench`.

In [27]:
stations = pd.read_parquet(PATH_INTERMEDIATE / "stations_cleaned.parquet")

In [28]:
stations_for_validation = stations.drop(columns=["geo_point_2d", "geo_shape", "class_en"])

In [29]:
stations_for_validation.columns

Index(['ptcarid', 'longnamefrench', 'longnamedutch'], dtype='str')

In [30]:
passengers_avg_station_names_merge = passengers_avg_sncb_2024.merge(
    stations_for_validation,
    left_on="stations_sncb",
    right_on="longnamefrench",
    how="left",
    indicator=True
)

In [31]:
passengers_avg_station_names_merge["_merge"].value_counts()

_merge
both          508
left_only      46
right_only      0
Name: count, dtype: int64

In [32]:
passengers_avg_station_names_merge = (
    passengers_avg_station_names_merge.drop(columns=["longnamefrench", "longnamedutch"])
)

In [33]:
passenger_count_station_name_orphans = (
    passengers_avg_station_names_merge.loc[passengers_avg_station_names_merge["_merge"] == "left_only"]
)
passenger_count_station_name_orphans.head()

,stations_sncb,passenger_avg_weekdays,passenger_avg_saturday,passenger_avg_sunday,ptcarid,_merge
17,ANTWERPEN-CAAL,38611,24567,20787,NaN,left_only
50,BERCHEM-ST-AG.-BERCHEM,911,275,281,NaN,left_only
56,BEVEREN,1326,520,459,NaN,left_only
69,BOITSFORT/BOSVOORDE,930,239,136,NaN,left_only
74,BOONDAEL/BOONDAAL,1009,178,159,NaN,left_only


In [34]:
orphan_station_names_sncb = passenger_count_station_name_orphans["stations_sncb"].to_list()


In [35]:
orphan_station_names_sncb

['ANTWERPEN-CAAL',
 'BERCHEM-ST-AG.-BERCHEM',
 'BEVEREN',
 'BOITSFORT/BOSVOORDE',
 'BOONDAEL/BOONDAAL',
 "BRAINE-l'ALLEUD",
 'BRU. AIRPORT - ZAVENTEM',
 'BRU.-CENT.',
 'BRU.-CHAP./KAP.',
 'BRU.-CONG.',
 'BRU.-LUXEMBG',
 'BRU.-MIDI/ZUID',
 'BRU.-NOORD/NORD',
 'BRU.-SCHUMAN',
 'BRU.-WEST/OUEST',
 'COMINES/KOMEN',
 'ENGHIEN/EDINGEN',
 'FEXHE-LE-HT-CLOCHER',
 'FOREST-EST/VORST-OOST',
 'FOREST-MIDI/VORST-ZUID',
 'GERMOIR/MOUTERIJ',
 'HAREN-ZUID/SUD',
 'LA ROCHE',
 'MORTSEL-DEURNESTEENWEG',
 'MORTSEL-OUDE-GOD',
 'MOUSCRON/MOESKROEN',
 'RONSE/RENAIX',
 'RUISBR.-SAUVEGARDE',
 'SCHAARBEEK/SCHAERBEEK',
 'ST-DENIJS-BOEKEL',
 'ST-DENIS-BOVESSE',
 'ST-GEN-RODE/RHODE-ST-GEN',
 'ST-GHISLAIN',
 'ST-GILLIS',
 'ST-JOB',
 'ST-JORIS-WEERT',
 'ST-KATELIJNE-WAVER',
 'ST-MARIABURG',
 'ST-MARTENS-BODEGEM',
 'ST-NIKLAAS',
 'ST-TRUIDEN',
 'TOUR ET TAXIS/THURN EN TAXIS',
 'UCCLE/UKKEL-CALEVOET',
 'UCCLE/UKKEL-STALLE',
 "VIVIER D'OIE/DIESDELLE",
 'WATERMAEL/WATERMAAL']

### 6.2.5.2. Separating Station DataFrames with and without IDs

To resolve the inconsistencies in the `stations_sncb` column, two distinct DataFrames are extracted from the merge results:  

1) `orphan_stations_sncb_subset`: the 46 orphan station names that require resolution.  

2) `station_names_with_ptcarid_subset`: the 508 station names with their assigned IDs.

An `original_name` column is added to `orphan_stations_sncb_subset` by copying the values of the `stations_sncb` column, as a record of its station names before transformation.

In [36]:
print(passengers_avg_station_names_merge.shape)
passengers_avg_station_names_merge.columns

(554, 6)


Index(['stations_sncb', 'passenger_avg_weekdays', 'passenger_avg_saturday',
       'passenger_avg_sunday', 'ptcarid', '_merge'],
      dtype='str')

In [37]:
orphan_stations_sncb_subset = (
    passengers_avg_station_names_merge.loc[passengers_avg_station_names_merge["ptcarid"].isnull()]
)

orphan_stations_sncb_subset["original_name"] = orphan_stations_sncb_subset["stations_sncb"]

print(orphan_stations_sncb_subset.columns)
print(orphan_stations_sncb_subset["_merge"].value_counts())
orphan_stations_sncb_subset.head()

Index(['stations_sncb', 'passenger_avg_weekdays', 'passenger_avg_saturday',
       'passenger_avg_sunday', 'ptcarid', '_merge', 'original_name'],
      dtype='str')
_merge
left_only     46
right_only     0
both           0
Name: count, dtype: int64


,stations_sncb,passenger_avg_weekdays,passenger_avg_saturday,passenger_avg_sunday,ptcarid,_merge,original_name
17,ANTWERPEN-CAAL,38611,24567,20787,NaN,left_only,ANTWERPEN-CAAL
50,BERCHEM-ST-AG.-BERCHEM,911,275,281,NaN,left_only,BERCHEM-ST-AG.-BERCHEM
56,BEVEREN,1326,520,459,NaN,left_only,BEVEREN
69,BOITSFORT/BOSVOORDE,930,239,136,NaN,left_only,BOITSFORT/BOSVOORDE
74,BOONDAEL/BOONDAAL,1009,178,159,NaN,left_only,BOONDAEL/BOONDAAL


In [38]:
station_names_with_ptcarid_subset = (
    passengers_avg_station_names_merge.loc[~passengers_avg_station_names_merge["ptcarid"].isnull()]
)

print(station_names_with_ptcarid_subset.columns)
print(station_names_with_ptcarid_subset["_merge"].value_counts())
station_names_with_ptcarid_subset.head()

Index(['stations_sncb', 'passenger_avg_weekdays', 'passenger_avg_saturday',
       'passenger_avg_sunday', 'ptcarid', '_merge'],
      dtype='str')
_merge
both          508
left_only       0
right_only      0
Name: count, dtype: int64


,stations_sncb,passenger_avg_weekdays,passenger_avg_saturday,passenger_avg_sunday,ptcarid,_merge
0,AALST,5883,2220,1872,6.0,both
1,AALST-KERREBROEK,15,<NA>,<NA>,104.0,both
2,AALTER,2348,622,620,8.0,both
3,AARSCHOT,5804,2243,1758,9.0,both
4,AARSELE,26,<NA>,<NA>,10.0,both


In [39]:
remaining_rows = (
    passengers_avg_station_names_merge.shape[0] - station_names_with_ptcarid_subset.shape[0]
    - orphan_stations_sncb_subset.shape[0]
)

print(f"Rows lost during DataFrame splitting: {remaining_rows}")

Rows lost during DataFrame splitting: 0


### 6.5.2.3. Resolving Orphan Station Names

In the section below:

* The resolution of the orphan station names is performed against `stations_for_validation`, which contains three reference columns:  
    * `ptcarid`: the target ID  
    * `longnamefrench` and `longnamedutch`: the French and Dutch station names used for matching against the `stations_sncb` column from `orphan_stations_sncb_subset`.

* The resolution uses the `ip.find_ptcarid()` function from the `infrabel_punctuality` package.

* Two dictionaries are created as required inputs for `ip.find_ptcarid()`:  
    1) `station_name_abbreviations`: maps abbreviations in `stations_sncb` to their full station names.
    2) `station_name_normalisation`: maps inconsistent `stations_sncb` names, i.e. names that do not follow an identifiable pattern, to their normalised version.

* The resolution proceeds in four steps: 
    1) First call of `ip.find_ptcarid()`:
        * **36** out of 46 **orphan stations** are matched with their correct station IDs, collected in the `stations_resolved` DataFrame.

        * **10** orphan stations remain unresolved.

    2) For **4** of these 10 orphan stations, a **manual lookup** in `stations_for_validation` is required to identify their exact matches.

    3) A **second normalised station name dictionary** `second_station_name_normalisation` is created to prepare the second call of the function.

    4) Second call of `ip.find_ptcarid()`: 
        * **9** out of 10 **remaining orphan stations** are now matched with their correct station IDs and collected in the `stations_resolved_2` DataFrame.

        * The **last orphan station**, `MORTSEL-DEURNESTEENWEG`, does not exist in the `stations_cleaned` reference dataset. Its case was analysed in the *02_04_profiling_and_cleaning_punctuality* notebook, and is detailed at the end of this section.

In [40]:
orphan_stations_sncb_subset = orphan_stations_sncb_subset.drop(columns=["_merge"])

In [41]:
station_name_abbreviations = {   
    "ST-" : "SINT-",
    "BRU." : "BRUXELLES",
    "CAAL" : "CENTRAAL",
    "CENT." : "CENTRAL",
    "CHAP." : "CHAPELLE",
    "CONG." : "CONGRES",
    "AG." : "AGATHE",
    "HT-" : "HAUT-",
    "LUXEMBG" : "LUXEMBOURG",
    "RUISBR." : "RUISBROEK",
    "GEN-" : "GENESIUS-"
   }

In [42]:
station_name_normalisation = {
    "BRUXELLES AIRPORT" : "BRUSSELS AIRPORT",
    "BRUXELLES-NOORD/NORD" : "BRUXELLES-NORD",
    "BRUXELLES-MIDI/ZUID" : "BRUXELLES-MIDI",
    "BRUXELLES-WEST/OUEST" : "BRUXELLES-OUEST",
    "UCCLE/UKKEL-CALEVOET" : "UCCLE-CALEVOET",
    "UCCLE/UKKEL-STALLE" : "UCCLE-STALLE"
}

In [43]:
ref_columns=["longnamedutch", "longnamefrench"]

In [44]:
stations_resolved, still_orphans = ip.find_ptcarid(
                            df_orphans=orphan_stations_sncb_subset,
                            df_ref=stations_for_validation,
                            orphan_column="stations_sncb",
                            ref_columns=ref_columns,
                            abbr_to_replace=station_name_abbreviations,
                            names_to_replace=station_name_normalisation,
                            id_column="ptcarid"
                    )


In [45]:
print(f"Station names harmonised after the first call of find_ptcarid(): {stations_resolved.shape[0]}")

print(f"Orphan station names still without station IDs after the first call of "
      f"find_ptcarid(): {still_orphans.shape[0]}")

Station names harmonised after the first call of find_ptcarid(): 36
Orphan station names still without station IDs after the first call of find_ptcarid(): 10


In [46]:
print("---Orphan Station Still to Resolve---")
still_orphans["stations_sncb"].to_list()

---Orphan Station Still to Resolve---


['BERCHEM-SINT-AGATHE-BERCHEM',
 'BEVEREN',
 'FORESINT-EST',
 'FORESINT-MIDI',
 'LA ROCHE',
 'MORTSEL-DEURNESTEENWEG',
 'MORTSEL-OUDE-GOD',
 'SINT-DENIS-BOVESSE',
 'SINT-GHISLAIN',
 'SINT-GILLIS']

In [47]:
stations_for_validation.loc[stations_for_validation["longnamedutch"].str.contains("GILLIS")]

,ptcarid,longnamefrench,longnamedutch
374,1730,SINT-GILLIS(DENDERMONDE),SINT-GILLIS(DENDERMONDE)


In [48]:
stations_for_validation.loc[stations_for_validation["longnamedutch"].str.contains("MORTSEL")]

,ptcarid,longnamefrench,longnamedutch
83,866,MORTSEL-OUDE GOD,MORTSEL-OUDE GOD
101,863,MORTSEL,MORTSEL
624,877,MORTSEL-LIERSESTEENWEG,MORTSEL-LIERSESTEENWEG


In [49]:
stations_for_validation.loc[stations_for_validation["longnamedutch"].str.contains("BEVEREN")]

,ptcarid,longnamefrench,longnamedutch
381,151,BEVEREN(WAAS),BEVEREN(WAAS)


In [50]:
stations_for_validation.loc[stations_for_validation["longnamedutch"].str.contains("LA ROCHE")]

,ptcarid,longnamefrench,longnamedutch
544,692,LA ROCHE(BRABANT),LA ROCHE(BRABANT)


In [51]:
second_station_name_normalisation = {
    "BERCHEM-SINT-AGATHE-BERCHEM" : "BERCHEM-SAINTE-AGATHE",
    "FORESINT-EST" : "FOREST-EST",
    "FORESINT-MIDI" : "FOREST-MIDI",
    "SINT-" : "SAINT-",
    "SAINT-GILLIS" : "SINT-GILLIS(DENDERMONDE)",
    "MORTSEL-OUDE-GOD" : "MORTSEL-OUDE GOD",
    "BEVEREN" : "BEVEREN(WAAS)",
    "LA ROCHE" : "LA ROCHE(BRABANT)"
}

In [52]:
stations_resolved_2, remaining_orphans = ip.find_ptcarid(
                            df_orphans=still_orphans,
                            df_ref=stations_for_validation,
                            orphan_column="stations_sncb",
                            ref_columns=ref_columns,
                            abbr_to_replace=station_name_abbreviations,
                            names_to_replace=second_station_name_normalisation,
                            id_column="ptcarid"
                    )

In [53]:
print(f"Station names harmonised after the second call of find_ptcarid(): {stations_resolved_2.shape[0]}")

print(f"Orphan station names still without station IDs after the second call of "
      f"find_ptcarid(): {remaining_orphans.shape[0]}")

Station names harmonised after the second call of find_ptcarid(): 9
Orphan station names still without station IDs after the second call of find_ptcarid(): 1


In [54]:
remaining_orphans["stations_sncb"].to_list()

['MORTSEL-DEURNESTEENWEG']

As explained in the *02_04_profiling_and_cleaning_punctuality* notebook, Section 4.2.5.4., `MORTSEL-DEURNESTEENWEG` is a discontinued station that merged into `MORTSEL` on 15 December 2024.  

As decided in the 02_04 notebook, an entry will be manually added to `stations_cleaned` durin the building of the `stations_dimension`, in the *03_01_building_dimension_station* notebook.

### 6.5.2.4. Concatenating Passenger Count Average Datasets with IDs

* In the last two sections, three DataFrames with harmonised station names and their matched station IDs have been created from `passengers_avg_sncb_2024`:  
    1) `station_names_with_ptcarid_subset`: **508 station names**  
    2) `stations_resolved`: **36 station names**  
    3) `stations_resolved_2`: **9 station names**   

* In addition, the `remaining_orphans` DataFrame still contains **1 station name**, `MORTSEL-DEURNESTEENWEG`, which does not exist in `stations_cleaned` and therefore has no station ID. As established earlier, this station will be manually added during the `stations_dimension` build in the *03_01_building_dimension_station* notebook.

In the section below, in order to create the `passengers_avg_sncb_2024_cleaned` dataset:

* The fixed value `"direct_match"` is assigned to the `resolution_method` column for the 508 rows of `station_names_with_ptcarid_subset`.

* The four DataFrames are concatenated into `passengers_avg_sncb_2024_cleaned`.  

* The `ptcarid` column is cast as `Int64` to prepare loading to SQL Server.

In [55]:
stations_id_1 = station_names_with_ptcarid_subset.shape[0]
stations_id_2 = stations_resolved.shape[0]
stations_id_3 = stations_resolved_2.shape[0]
print(f"Total Stations with ptcarid: {stations_id_1 + stations_id_2 + stations_id_3}")

Total Stations with ptcarid: 553


In [56]:
station_names_with_ptcarid_subset.columns

Index(['stations_sncb', 'passenger_avg_weekdays', 'passenger_avg_saturday',
       'passenger_avg_sunday', 'ptcarid', '_merge'],
      dtype='str')

In [57]:
print(stations_resolved.columns)
stations_resolved_2.columns

Index(['stations_sncb', 'passenger_avg_weekdays', 'passenger_avg_saturday',
       'passenger_avg_sunday', 'original_name', 'ptcarid',
       'resolution_method'],
      dtype='str')


Index(['stations_sncb', 'passenger_avg_weekdays', 'passenger_avg_saturday',
       'passenger_avg_sunday', 'original_name', 'ptcarid',
       'resolution_method'],
      dtype='str')

In [58]:
station_names_with_ptcarid_subset = station_names_with_ptcarid_subset.drop(columns=["_merge"])

In [59]:
station_names_with_ptcarid_subset["resolution_method"] = "direct_match"

In [60]:
passengers_avg_sncb_2024_cleaned = pd.concat([
    station_names_with_ptcarid_subset,
    stations_resolved,
    stations_resolved_2, 
    remaining_orphans
], ignore_index=True)

In [61]:
passengers_avg_sncb_2024_cleaned["ptcarid"] = (
    passengers_avg_sncb_2024_cleaned["ptcarid"].astype("Int64")
)

The output below confirms the expected missing values:  
* `MORTSEL-DEURNESTEENWEG` orphan station: no `ptcarid` value & no `resolution_method` value  

* Stations closed on weekdays, Saturdays, or Sundays  

* No `original_name` values for stations matched directly against `stations_cleaned`  

In [62]:
passengers_avg_sncb_2024_cleaned.isnull().sum()

stations_sncb               0
passenger_avg_weekdays      2
passenger_avg_saturday     45
passenger_avg_sunday       44
ptcarid                     1
resolution_method           1
original_name             508
dtype: int64

In [63]:
passengers_avg_sncb_2024_cleaned.info()

<class 'pandas.DataFrame'>
RangeIndex: 554 entries, 0 to 553
Data columns (total 7 columns):
 #   Column                  Non-Null Count  Dtype 
---  ------                  --------------  ----- 
 0   stations_sncb           554 non-null    object
 1   passenger_avg_weekdays  552 non-null    Int64 
 2   passenger_avg_saturday  509 non-null    Int64 
 3   passenger_avg_sunday    510 non-null    Int64 
 4   ptcarid                 553 non-null    Int64 
 5   resolution_method       553 non-null    str   
 6   original_name           46 non-null     object
dtypes: Int64(4), object(2), str(1)
memory usage: 39.3+ KB


## 6.3. Export to Silver Layer

The `passengers_avg_sncb_2024` DataFrame is now ready to be exported to the **intermediate directory** as `passengers_avg_sncb_2024_cleaned.parquet`.

In [64]:
passengers_avg_sncb_2024_cleaned.to_parquet(PATH_INTERMEDIATE / "passengers_avg_sncb_2024_cleaned.parquet")

In [65]:
df_to_verify = pd.read_parquet(PATH_INTERMEDIATE / "passengers_avg_sncb_2024_cleaned.parquet")

In [66]:
print(df_to_verify.shape, df_to_verify.dtypes.to_dict())
df_to_verify.head()

(554, 7) {'stations_sncb': <StringDtype(na_value=nan)>, 'passenger_avg_weekdays': Int64Dtype(), 'passenger_avg_saturday': Int64Dtype(), 'passenger_avg_sunday': Int64Dtype(), 'ptcarid': Int64Dtype(), 'resolution_method': <StringDtype(na_value=nan)>, 'original_name': <StringDtype(na_value=nan)>}


,stations_sncb,passenger_avg_weekdays,passenger_avg_saturday,passenger_avg_sunday,ptcarid,resolution_method,original_name
0,AALST,5883,2220,1872,6,direct_match,NaN
1,AALST-KERREBROEK,15,<NA>,<NA>,104,direct_match,NaN
2,AALTER,2348,622,620,8,direct_match,NaN
3,AARSCHOT,5804,2243,1758,9,direct_match,NaN
4,AARSELE,26,<NA>,<NA>,10,direct_match,NaN
